# base_rh.ipynb — Semana 04 · Dia 01
## Leitura de arquivos + módulo datetime
**SENAI · Visualização de Dados e BI · Turma T2**

---

### Como usar este notebook
- Execute cada célula na ordem, uma por vez (**Shift + Enter**)
- Leia os comentários antes de rodar — eles explicam o que acontece
- Se uma célula der erro, leia a mensagem: ela diz exatamente o que faltou
- Ao final, todo o código deste notebook vira o seu `base_rh.py`

---

### O que você vai aprender hoje
1. Como importar bibliotecas no Python
2. Como ler um CSV com pandas — e por que os parâmetros importam
3. Os comandos básicos de exploração de um DataFrame
4. Como trabalhar com datas no Python
5. Como exportar dados para JSON e Excel
6. Como criar funções para reutilizar código

---

> 🏭 **Contexto real:** você foi contratado como analista de dados júnior em uma indústria de Jaraguá do Sul. O time de RH mandou a base de funcionários. Antes de qualquer análise, você precisa entender o que veio.

---
## 📦 CÉLULA 1 — Importando as bibliotecas

**O que é uma biblioteca?**  
É um conjunto de ferramentas prontas que alguém já escreveu para você usar.  
Você não precisa criar tudo do zero — só importar e usar.

| Biblioteca | Para que serve | Como importar |
|---|---|---|
| `pandas` | Ler, transformar e analisar dados em tabelas | `import pandas as pd` |
| `re` | Encontrar e substituir padrões em texto (regex) | `import re` |
| `datetime` | Trabalhar com datas e horas | `from datetime import datetime` |

**Por que `as pd`?**  
É um apelido. Em vez de escrever `pandas.read_csv()` toda vez, você escreve `pd.read_csv()`.  
É uma convenção do mercado — todo analista usa `pd` para pandas.

In [ ]:
# Sempre a primeira célula do notebook: importar o que você vai usar
# Se aparecer erro aqui, a biblioteca não está instalada.
# Solução: abra o terminal e rode:  pip install pandas openpyxl

import pandas as pd      # biblioteca de análise de dados
import re                # expressões regulares (para limpar texto)
from datetime import datetime  # para trabalhar com datas

# Confirmar que as importações funcionaram
print('✓ pandas versão:', pd.__version__)
print('✓ Bibliotecas importadas com sucesso!')

---
## 📂 CÉLULA 2 — Lendo o arquivo CSV

**O que é um CSV?**  
É um arquivo de texto onde os dados ficam separados por um caractere (vírgula, ponto-e-vírgula etc.)  
CSV = *Comma-Separated Values* = Valores Separados por Vírgula

**Por que precisamos de parâmetros especiais?**  
O arquivo `base_rh.csv` foi gerado pelo **TOTVS** — sistema ERP muito usado em indústrias da região.  
O TOTVS salva arquivos no padrão europeu/Windows, que é diferente do padrão internacional:

| Parâmetro | Por que usar | O que acontece se não usar |
|---|---|---|
| `sep=";"` | O TOTVS separa colunas com `;` | O pandas lê tudo como 1 coluna só |
| `encoding="cp1252"` | Encoding do Windows — suporta ã, é, ç | "Produção" vira "Produ??o" |
| `decimal=","` | No Brasil o decimal é vírgula: 9.088,34 | Salário vira texto, não número |

In [ ]:
# pd.read_csv() lê o arquivo e cria um DataFrame (uma tabela dentro do Python)
# CAMINHO = caminho completo para o arquivo CSV

CAMINHO = r'C:\Users\s2bcl\OneDrive\Documents\Lab365\turma\visualizacao_de_dados\turma-visualizacao-de-dados\aulas\semana_04\base_rh.csv'

df = pd.read_csv(
    CAMINHO,
    sep=';',           # separador de colunas
    encoding='cp1252', # encoding do Windows
    decimal=','        # vírgula como separador decimal
)

print('✓ Arquivo lido com sucesso!')
print(f'  Total de linhas  : {df.shape[0]}')
print(f'  Total de colunas : {df.shape[1]}')

**O que é `df.shape`?**  
`shape` é uma propriedade do DataFrame que retorna uma tupla `(linhas, colunas)`.  
- `df.shape[0]` → número de linhas (registros)
- `df.shape[1]` → número de colunas (campos)

Você vai usar isso em **todo projeto** para confirmar que leu o arquivo certo.

---
## 🔍 CÉLULA 3 — Primeiras olhadas no dado

Antes de mexer em qualquer coisa, você precisa **ver** o que chegou.  
Esses três comandos são os mais usados no dia a dia de um analista.

In [ ]:
# df.head(n) → mostra as PRIMEIRAS n linhas do DataFrame
# Padrão: head(5) se você não passar nenhum número
# Use para ter uma rápida visão geral do conteúdo

df.head(5)

In [ ]:
# df.tail(n) → mostra as ÚLTIMAS n linhas
# Serve para confirmar que o arquivo foi lido até o fim
# Se o arquivo tiver 1.000 linhas, o tail deve mostrar IDs próximos de 1000

df.tail(3)

In [ ]:
# df.columns → lista com o nome de todas as colunas
# .tolist() converte o resultado para uma lista Python comum

print('Colunas do dataset:')
print(df.columns.tolist())

---
## 🩺 CÉLULA 4 — Diagnóstico: tipos e nulos

**O que são tipos de dados?**  
Cada coluna armazena um tipo de informação. O Python precisa saber o tipo  
para saber o que pode fazer com ela:

| Tipo | O que é | Exemplo |
|---|---|---|
| `int64` | Número inteiro | 1, 42, 1000 |
| `float64` | Número decimal | 9088.34, 3.14 |
| `object` | Texto / string | "Julia Nunes", "Ativo" |
| `datetime64` | Data e hora | 2024-08-13 |

**O que são nulos?**  
São células vazias — o campo não foi preenchido no sistema de origem.  
Em Python/pandas, nulos aparecem como `NaN` (*Not a Number*) ou `NaT` (*Not a Time*).

In [ ]:
# df.info() → resumo completo do DataFrame em uma só chamada:
# - Nome de cada coluna
# - Quantidade de valores não-nulos
# - Tipo de dado (dtype)
# - Uso de memória
# É o comando mais importante para o primeiro diagnóstico!

df.info()

In [ ]:
# df.isnull() → cria uma tabela de True/False (True = é nulo)
# .sum()      → soma os True de cada coluna (True vale 1, False vale 0)
# Resultado: quantidade de nulos por coluna

print('Quantidade de nulos por coluna:')
print(df.isnull().sum())

print()

# Percentual de nulos — mais útil em bases grandes
# len(df) = número total de linhas
print('Percentual de nulos por coluna:')
print((df.isnull().sum() / len(df) * 100).round(2))

---
## 📊 CÉLULA 5 — Estatísticas das colunas numéricas

`describe()` gera um resumo estatístico automático.  
Você vai usar isso para detectar valores suspeitos (ex: salário de R$0,00 ou idade de 200 anos).

| Estatística | O que significa |
|---|---|
| `count` | Quantos valores não-nulos existem |
| `mean` | Média |
| `std` | Desvio padrão — o quanto os valores variam em torno da média |
| `min` | Menor valor |
| `25%` | 25% dos registros estão abaixo desse valor |
| `50%` | Metade dos registros estão abaixo (mediana) |
| `75%` | 75% dos registros estão abaixo desse valor |
| `max` | Maior valor |

In [ ]:
# df.describe() → estatísticas das colunas numéricas
# .round(2)    → arredonda para 2 casas decimais

df.describe().round(2)

---
## 🗂️ CÉLULA 6 — Explorando colunas categóricas

**Coluna categórica** = coluna com poucos valores possíveis (departamento, cargo, status...)  

`nunique()` conta quantos valores diferentes existem.  
`value_counts()` mostra a distribuição — quantas vezes cada valor aparece.

In [ ]:
# df.nunique() → retorna quantos valores ÚNICOS cada coluna tem
# Serve para identificar quais são categóricas (poucas opções)
# e quais são identificadoras (muitas opções diferentes)

print('Valores únicos por coluna:')
print(df.nunique())

In [ ]:
# df['coluna'].value_counts() → conta quantas vezes cada valor aparece,
# ordenado do mais frequente para o menos frequente

# Vamos ver todas as colunas categóricas de uma vez usando um loop:
# 'for col in lista' = para cada coluna na lista, executa o bloco abaixo

for col in ['Departamento', 'Cargo', 'Genero', 'Status', 'Estado_Civil']:
    print(f'\n── {col} ──')   # f-string: {} insere o valor da variável
    print(df[col].value_counts())

---
## 🔎 CÉLULA 7 — Selecionando colunas e filtrando linhas

São as operações mais usadas no dia a dia. Você vai usar isso em todo projeto.

**Seleção de coluna** → `df['Nome']`  
**Seleção de múltiplas colunas** → `df[['Nome', 'Salario']]`  
**Filtro de linhas** → `df[df['Status'] == 'Ativo']`

In [ ]:
# Selecionar UMA coluna → retorna uma Series (lista com índice)
# Uma Series é como uma coluna de planilha isolada

df['Nome'].head(5)

In [ ]:
# Selecionar MÚLTIPLAS colunas → retorna um DataFrame
# Note os dois colchetes: df[  ['col1', 'col2']  ]
# O primeiro [] é o operador de acesso do DataFrame
# O segundo [] é a lista de colunas

df[['Nome', 'Departamento', 'Cargo', 'Salario']].head(6)

In [ ]:
# FILTRO SIMPLES: selecionar só as linhas onde Status == 'Ativo'
# df['Status'] == 'Ativo' → cria uma Series de True/False
# df[ True/False ]        → mantém só as linhas onde é True

ativos = df[df['Status'] == 'Ativo']

print(f'Total de funcionários : {len(df)}')
print(f'Funcionários ativos   : {len(ativos)}')
print(f'Funcionários inativos : {len(df) - len(ativos)}')

In [ ]:
# FILTRO COMPOSTO: duas condições ao mesmo tempo
# Use & para AND (as duas condições precisam ser verdadeiras)
# Use | para OR  (pelo menos uma precisa ser verdadeira)
# IMPORTANTE: cada condição precisa estar entre parênteses

gerentes_ti = df[
    (df['Cargo'] == 'Gerente') & (df['Departamento'] == 'TI')
]

print(f'Gerentes na área de TI: {len(gerentes_ti)}')
print()
gerentes_ti[['Nome', 'Salario', 'Status', 'Data_Admissao']]

In [ ]:
# Estatísticas rápidas de uma coluna numérica
# .mean()  → média
# .max()   → maior valor
# .min()   → menor valor
# .median()→ mediana (valor do meio quando ordenados)

print(f"Salário médio : R$ {df['Salario'].mean():,.2f}")
print(f"Salário máximo: R$ {df['Salario'].max():,.2f}")
print(f"Salário mínimo: R$ {df['Salario'].min():,.2f}")
print(f"Salário mediana:R$ {df['Salario'].median():,.2f}")

# :,.2f → formata o número com separador de milhar (,) e 2 casas decimais (.2f)

---
## 📁 CÉLULA 8 — Exportando para JSON

**O que é JSON?**  
É o formato padrão de troca de dados entre sistemas modernos — APIs, ERPs na nuvem, dashboards web.  
JSON = *JavaScript Object Notation*

```json
[
  { "Nome": "Julia Nunes", "Departamento": "Logística", "Salario": 9088.34 },
  { "Nome": "Gustavo Duarte", "Departamento": "TI", "Salario": 8155.98 }
]
```

**Por que `orient="records"`?**  
Define o formato do JSON. `records` = cada linha vira um objeto separado.  
É o formato que APIs REST esperam receber.

In [ ]:
import json  # biblioteca nativa do Python para manipular JSON

# df.to_json() → converte o DataFrame para JSON e salva no arquivo
df.to_json(
    'base_rh.json',
    orient='records',      # cada linha vira um objeto {}
    force_ascii=False,     # False = salva ã, é, ç como estão
    indent=2               # 2 espaços de indentação → arquivo legível
)

print('✓ base_rh.json gerado!')

# Ler de volta para confirmar
df_json = pd.read_json('base_rh.json')
print(f'  Linhas lidas do JSON: {len(df_json)}')

# Visualizar os 2 primeiros registros no formato JSON
with open('base_rh.json', 'r', encoding='utf-8') as f:
    amostra = json.load(f)
print('\nPrimeiros 2 registros no JSON:')
print(json.dumps(amostra[:2], ensure_ascii=False, indent=2))

---
## 📊 CÉLULA 9 — Exportando para Excel

Excel ainda é o formato mais pedido por gestores em ambientes industriais.  
O pandas permite criar um arquivo com **múltiplas abas** — uma por departamento.

**O que é `ExcelWriter`?**  
É um gerenciador que permite escrever várias abas no mesmo arquivo `.xlsx`.  
O `with ... as writer:` garante que o arquivo seja fechado corretamente ao final.

> ⚠️ Precisa do `openpyxl` instalado: `pip install openpyxl`

In [ ]:
# Exportação simples: tudo em uma aba
df.to_excel('base_rh.xlsx', index=False, sheet_name='Funcionarios')
print('✓ base_rh.xlsx (aba única) gerado!')

# Exportação avançada: uma aba por departamento
with pd.ExcelWriter('base_rh_por_depto.xlsx', engine='openpyxl') as writer:

    # Aba 1: dataset completo
    df.to_excel(writer, sheet_name='Todos', index=False)
    print('  Aba Todos: todos os registros')

    # df['Departamento'].unique() → lista de departamentos sem repetição
    # sorted() → ordena em ordem alfabética
    for depto in sorted(df['Departamento'].unique()):

        # Filtrar só os funcionários desse departamento
        df_depto = df[df['Departamento'] == depto]

        # Limite do Excel: nomes de aba têm no máximo 31 caracteres
        nome_aba = depto[:31]

        df_depto.to_excel(writer, sheet_name=nome_aba, index=False)
        print(f'  Aba "{nome_aba}": {len(df_depto)} registros')

print('\n✓ base_rh_por_depto.xlsx gerado!')

---
## ☕ INTERVALO — 20h30

**Antes de continuar, confirme:**
- [ ] Os arquivos `base_rh.json` e `base_rh.xlsx` apareceram na sua pasta?
- [ ] O `df.shape` mostrou `(1000, 10)`?
- [ ] Você entendeu a diferença entre `head()` e `tail()`?

Se algum arquivo não apareceu, chame o professor agora — antes de continuar.

---
## 📅 CÉLULA 10 — Módulo datetime: o básico

**Por que isso importa?**  
A coluna `Data_Admissao` veio como **texto** (`object`).  
O Python não sabe que `'13/08/2024'` é uma data — pra ele é só uma string.  
Enquanto for texto, você **não consegue** calcular diferenças, filtrar por período, extrair mês ou ano.

**Dois comandos essenciais:**

| Comando | Significado | Exemplo |
|---|---|---|
| `strptime` | string → parse → time (lê um texto e vira data) | `'13/08/2024'` → `datetime(2024, 8, 13)` |
| `strftime` | string → format → time (pega uma data e formata como texto) | `datetime(2024, 8, 13)` → `'Agosto de 2024'` |

**Códigos de formato de data:**
| Código | Significado | Exemplo |
|---|---|---|
| `%d` | Dia com zero à esquerda | 08 |
| `%m` | Mês com zero à esquerda | 08 |
| `%Y` | Ano com 4 dígitos | 2024 |
| `%B` | Nome do mês por extenso | August |

In [ ]:
from datetime import datetime, timedelta

# ── strptime: texto → objeto data ─────────────────────────────────────────
# Pegamos o primeiro valor da coluna Data_Admissao como exemplo
data_texto = '13/08/2024'

# datetime.strptime(texto, formato)
# O formato descreve COMO a string está escrita:
# '%d/%m/%Y' = dia/mês/ano
data_obj = datetime.strptime(data_texto, '%d/%m/%Y')

print(f'String original  : "{data_texto}"')
print(f'Objeto datetime  : {data_obj}')
print(f'Tipo             : {type(data_obj)}')
print()

# ── strftime: objeto data → texto formatado ───────────────────────────────
# Agora podemos formatar a data do jeito que quisermos
formatado = data_obj.strftime('%d de %B de %Y')  # 13 de August de 2024
print(f'Formatado        : {formatado}')

# Extraindo partes da data
print(f'Ano              : {data_obj.year}')
print(f'Mês              : {data_obj.month}')
print(f'Dia              : {data_obj.day}')

**O que é `timedelta`?**  
É o resultado da subtração entre duas datas — a diferença de tempo.  
Se você subtrair uma data de outra, recebe um `timedelta` com a diferença em dias.

In [ ]:
# ── timedelta: diferença entre duas datas ────────────────────────────────
# datetime.today() → data e hora de agora
hoje = datetime.today()

# Subtrair duas datas retorna um objeto timedelta
diferenca = hoje - data_obj

print(f'Data de admissão : {data_texto}')
print(f'Hoje             : {hoje.strftime("%d/%m/%Y")}')
print(f'Diferença        : {diferenca.days} dias')
print(f'Em anos          : {diferenca.days / 365.25:.1f} anos')

# Por que 365.25 e não 365?
# Todo 4 anos existe um ano bissexto com 366 dias.
# 365.25 é a média correta: (365 × 3 + 366) / 4 = 365.25

---
## 📅 CÉLULA 11 — Convertendo a coluna inteira com pd.to_datetime

A célula anterior mostrou como converter **uma** string em data.  
Mas nosso DataFrame tem 1.000 linhas.  

`pd.to_datetime()` converte a coluna **inteira de uma vez** — sem loop, muito mais eficiente.

In [ ]:
# Antes da conversão:
print(f'Tipo ANTES : {df["Data_Admissao"].dtype}')
print(f'Valor antes: "{df["Data_Admissao"].iloc[0]}"')
# .iloc[0] → acessa a primeira linha pelo índice (iloc = integer location)

print()

# Converter a coluna inteira
# Trabalhamos em uma cópia para não alterar o df original
df_limpo = df.copy()
# .copy() → cria uma cópia independente
# Sem isso, mudanças em df_limpo afetariam o df original

df_limpo['Data_Admissao'] = pd.to_datetime(
    df_limpo['Data_Admissao'],
    format='%d/%m/%Y',  # formato da string de entrada
    errors='coerce'     # datas inválidas viram NaT (Not a Time) em vez de erro
)

print(f'Tipo DEPOIS: {df_limpo["Data_Admissao"].dtype}')
print(f'Valor depois: {df_limpo["Data_Admissao"].iloc[0]}')
print(f'NaT gerados : {df_limpo["Data_Admissao"].isna().sum()}')

---
## 📅 CÉLULA 12 — Extraindo informações da data com o acessor `.dt`

Depois de converter para datetime, o pandas libera o **acessor `.dt`**  
que dá acesso a propriedades da data:

| Código | O que retorna | Exemplo |
|---|---|---|
| `.dt.year` | Ano | 2024 |
| `.dt.month` | Mês (número) | 8 |
| `.dt.quarter` | Trimestre | 3 |
| `.dt.day_name()` | Nome do dia da semana | "Tuesday" |

In [ ]:
# Criar colunas derivadas de data
df_limpo['Ano_Admissao'] = df_limpo['Data_Admissao'].dt.year
df_limpo['Mes_Admissao'] = df_limpo['Data_Admissao'].dt.month
df_limpo['Trimestre']    = df_limpo['Data_Admissao'].dt.quarter

# Calcular tempo de casa em anos
# pd.Timestamp.today() → data atual no formato do pandas
hoje_ts = pd.Timestamp.today()

df_limpo['Tempo_Casa_Anos'] = (
    (hoje_ts - df_limpo['Data_Admissao']).dt.days / 365.25
).round(1)
# Subtração de datas → timedelta
# .dt.days → extrai a diferença em dias
# / 365.25 → converte dias em anos
# .round(1) → arredonda para 1 casa decimal

# Ver o resultado
cols = ['Nome', 'Data_Admissao', 'Ano_Admissao', 'Mes_Admissao', 'Trimestre', 'Tempo_Casa_Anos']
df_limpo[cols].head(6)

---
## ⚙️ CÉLULA 13 — Criando sua primeira função

**O que é uma função?**  
É um bloco de código com nome próprio que você pode reutilizar.  
Em vez de copiar o mesmo código 10 vezes, você define uma vez e chama quando precisar.

```python
def nome_da_funcao(parametro):
    # o que a função faz
    return resultado
```

- `def` → palavra-chave que define uma função
- `parametro` → valor de entrada que a função recebe
- `return` → valor de saída que a função entrega
- A indentação (4 espaços) define o que está dentro da função

In [ ]:
# Função que classifica o tempo de casa em uma categoria de texto
# Por que criar isso? Categorias são mais fáceis de usar em gráficos
# e filtros no Power BI do que números decimais.

def classificar_tempo(anos):
    """
    Recebe: anos (número) — o tempo de casa do funcionário
    Retorna: uma categoria em texto
    """
    # pd.isna() verifica se o valor é nulo (NaN ou NaT)
    if pd.isna(anos):
        return 'Sem data'
    elif anos < 1:
        return 'Menos de 1 ano'
    elif anos < 3:
        return '1 a 3 anos'
    elif anos < 5:
        return '3 a 5 anos'
    else:
        return 'Mais de 5 anos'


# Testar a função com valores individuais antes de aplicar no DataFrame
print('Testando a função:')
print(f'  0.5 anos → "{classificar_tempo(0.5)}"')
print(f'  2.1 anos → "{classificar_tempo(2.1)}"')
print(f'  4.8 anos → "{classificar_tempo(4.8)}"')
print(f'  9.0 anos → "{classificar_tempo(9.0)}"')

In [ ]:
# .apply(função) → aplica a função em cada linha da coluna
# É como fazer um loop em cada célula, mas de forma mais eficiente

df_limpo['Faixa_Tempo_Casa'] = df_limpo['Tempo_Casa_Anos'].apply(classificar_tempo)

print('Distribuição por faixa de tempo de casa:')
print(df_limpo['Faixa_Tempo_Casa'].value_counts())

---
## 📆 CÉLULA 14 — Filtrando por período de datas

Agora que a data é um tipo real, podemos fazer filtros por período.  
`pd.DateOffset(years=1)` → um deslocamento de exatamente 1 ano no calendário.  
É mais preciso que `365 dias` porque considera meses com 28/30/31 dias e anos bissextos.

In [ ]:
hoje = pd.Timestamp.today()

# Definir os pontos de corte
um_ano_atras    = hoje - pd.DateOffset(years=1)
cinco_anos_atras = hoje - pd.DateOffset(years=5)

# Filtrar por período
# >= significa 'maior ou igual a' → admitidos depois de 1 ano atrás
novos     = df_limpo[df_limpo['Data_Admissao'] >= um_ano_atras]
veteranos = df_limpo[df_limpo['Data_Admissao'] <  cinco_anos_atras]

print(f'Admitidos nos últimos 12 meses : {len(novos)} funcionários')
print(f'Veteranos (5+ anos de casa)    : {len(veteranos)} funcionários')

print()

# Comparativo de salário: novos vs veteranos
print('Salário médio:')
print(f'  Novos     : R$ {novos["Salario"].mean():,.2f}')
print(f'  Veteranos : R$ {veteranos["Salario"].mean():,.2f}')

---
## 📊 CÉLULA 15 — Agrupamentos com groupby

**O que é `groupby`?**  
É como a tabela dinâmica do Excel — agrupa as linhas por uma coluna  
e aplica um cálculo (média, soma, contagem...) em cada grupo.

```
df.groupby('coluna_agrupamento')['coluna_calculo'].função()
```

In [ ]:
# Salário médio por departamento
# .mean()             → calcula a média de cada grupo
# .round(2)           → arredonda para 2 casas
# .sort_values(...)   → ordena do maior para o menor
# ascending=False     → decrescente (False = do maior para o menor)

print('Salário médio por departamento:')
print(
    df_limpo.groupby('Departamento')['Salario']
    .mean()
    .round(2)
    .sort_values(ascending=False)
)

In [ ]:
# Tempo médio de casa por departamento
media_depto = (
    df_limpo
    .groupby('Departamento')['Tempo_Casa_Anos']
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .rename('Media_Anos')   # .rename() → renomeia a coluna resultante
    .reset_index()          # .reset_index() → converte o índice em coluna comum
)

print('Tempo médio de casa por departamento:')
print(media_depto.to_string(index=False))

# .idxmax() → retorna o NOME do índice com maior valor
maior = media_depto.set_index('Departamento')['Media_Anos'].idxmax()
menor = media_depto.set_index('Departamento')['Media_Anos'].idxmin()
print(f'\n→ Maior retenção: {maior}')
print(f'→ Menor retenção: {menor}')

In [ ]:
# Admissões por ano — histórico completo
# .size() → conta o número de linhas em cada grupo
# .sort_index() → ordena pelo índice (neste caso, o ano)

por_ano = (
    df_limpo
    .groupby('Ano_Admissao')
    .size()
    .rename('Admissoes')
    .reset_index()
    .dropna()  # remove linhas com Ano_Admissao nulo
)

print('Admissões por ano:')
for _, row in por_ano.iterrows():
    # .iterrows() → itera linha por linha, retornando (índice, Series)
    # _ → descartamos o índice (convenção: _ = 'não preciso desse valor')
    barra = '█' * int(row['Admissoes'] / 5)
    print(f'  {int(row["Ano_Admissao"])}  {row["Admissoes"]:>4}  {barra}')

---
## 💾 CÉLULA 16 — Salvando o dataset limpo e enriquecido

Aqui fazemos o **Load** do ETL: salvamos o resultado final  
em um arquivo CSV que vai alimentar o Power BI na próxima semana.

In [ ]:
# Salvar o DataFrame enriquecido
# encoding='utf-8-sig' → UTF-8 com BOM — necessário para o Excel no Windows
#                         abrir o arquivo sem distorcer ã, é, ç

df_limpo.to_csv(
    'base_rh_dia01.csv',
    index=False,         # não salva o índice numérico como coluna extra
    sep=';',             # separador ponto-e-vírgula
    encoding='utf-8-sig'
)

print('✓ base_rh_dia01.csv salvo!')
print(f'  Colunas finais: {df_limpo.columns.tolist()}')
print(f'  Linhas        : {len(df_limpo)}')

---
## 🏁 CÉLULA 17 — Exercício final

Coloque em prática tudo que você aprendeu hoje.  
Resolva os passos abaixo **nesta célula** (pode criar mais células se quiser).

---

### 🏭 Cenário
O gestor de RH da Indústria Metalúrgica Catarinense pediu 2 entregáveis:
1. A base de funcionários em Excel com **uma aba por departamento**
2. Um relatório de tempo de casa pronto para o Power BI

---

### Passos

**1.** Carregue o `base_rh.csv` com os parâmetros corretos e confirme que `Salario` é `float64`.

**2.** Use `df.info()` e `df.describe()` para fazer o diagnóstico. Anote em uma célula Markdown:  
→ *"Os 3 problemas mais importantes que encontrei foram..."*

**3.** Converta `Data_Admissao` para datetime. Crie as colunas:  
`Ano_Admissao`, `Mes_Admissao`, `Tempo_Casa_Anos` e `Faixa_Tempo_Casa`.

**4.** Responda com `groupby`:  
→ Qual departamento tem a **maior** média de tempo de casa?  
→ Qual tem a **menor**?

**5.** Exporte `base_rh_deptos.xlsx` com uma aba `'Completo'` e mais uma aba por departamento.

**6.** Faça o commit:  
```bash
git pull origin master
git add alunos/seunome/
git commit -m "semana 04 - dia 01: entrega arquivos e datetime"
git push origin master
git log --oneline -3
```
Cole o resultado do `git log` na última célula do notebook.

---

> 💡 **Dica para o Passo 4:**  
> `df.groupby('Departamento')['Tempo_Casa_Anos'].mean().sort_values(ascending=False)`

> ⚠️ **Se o push falhar com 'rejected':** você esqueceu o `git pull` antes.

In [ ]:
# ── Resolva o exercício aqui ──────────────────────────────────────────────
# Passo 1: carregue o CSV


# Passo 2: diagnóstico


# Passo 3: converter datas e criar colunas derivadas


# Passo 4: groupby — maior e menor média de tempo de casa


# Passo 5: exportar para Excel


# Passo 6: cole o output do git log aqui como comentário
# git log:

---
## ✅ Resumo do Dia 01

| Você aprendeu | Comando |
|---|---|
| Importar bibliotecas | `import pandas as pd` |
| Ler CSV com parâmetros | `pd.read_csv(..., sep, encoding, decimal)` |
| Ver o dado | `df.head()` · `df.tail()` · `df.info()` · `df.describe()` |
| Contar nulos | `df.isnull().sum()` |
| Ver categorias | `df.nunique()` · `df.value_counts()` |
| Filtrar linhas | `df[df['col'] == 'valor']` |
| Filtro duplo | `df[(cond1) & (cond2)]` |
| Converter data | `pd.to_datetime(..., format, errors)` |
| Extrair partes da data | `.dt.year` · `.dt.month` · `.dt.quarter` |
| Calcular diferença de datas | `(hoje - data).dt.days / 365.25` |
| Filtrar por período | `df[df['data'] >= um_ano_atras]` |
| Criar função | `def nome(param): ... return resultado` |
| Aplicar função na coluna | `.apply(função)` |
| Agrupar e calcular | `df.groupby('col')['col2'].mean()` |
| Exportar JSON | `df.to_json(..., orient='records')` |
| Exportar Excel | `pd.ExcelWriter` + `df.to_excel()` |

---

## 🔜 Próxima aula — Quinta · 19h

Na quinta vamos usar **Expressões Regulares** para limpar os dados de texto —  
incluindo remover aqueles títulos `Sr.`, `Dr.`, `Srta.` dos nomes de funcionários.  
O código que você escrever aqui hoje vai ser importado como módulo no Dia 03.

---

**📌 Entrega no AVA:** pasta `alunos/seunome/semana_04/` com `base_rh.ipynb`,  
`base_rh.json`, `base_rh_deptos.xlsx` e `base_rh_dia01.csv`. Prazo conforme AVA.